# LeetCode 567 — Permutation in String bằng Python

Link đề: https://leetcode.com/problems/permutation-in-string/ · Độ khó: **Medium** · Chủ đề: Hash Table, Two Pointers, String, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        ...
```

Ghi chú: bài này chỉ dùng chuỗi, không cần cấu trúc tự dựng. Mọi lời giải đều được đối chiếu với một oracle độc lập: sinh **mọi hoán vị** của `s1` bằng `itertools.permutations` rồi kiểm tra có xuất hiện trong `s2` hay không.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: "`s2` chứa một **hoán vị** của `s1`" nghĩa là có một **chuỗi con liên tiếp** (substring) của `s2` mà các chữ cái là sự sắp xếp lại của `s1` (cùng loại chữ, cùng số lần xuất hiện).
2. Nắm các công cụ Python cần thiết: cắt chuỗi `s[i:i+m]`, `sorted(...)`, `collections.Counter`, mảng đếm 26 ô với `ord(c) - ord('a')`, so sánh list, và `itertools.permutations` để làm oracle.
3. Đi từ **sắp xếp từng cửa sổ** (`O(n · m log m)`) → **đếm lại từng cửa sổ bằng `Counter`** (`O(n · m)`) → **cửa sổ trượt độ dài cố định, so sánh hai mảng đếm** (`O(26 · n)`) → **cửa sổ trượt có biến `diff`** (`O(n)`, lời giải nộp) → **cửa sổ co giãn hai con trỏ** (`O(n)`).
4. Hiểu mấu chốt: hai chuỗi là hoán vị của nhau khi và chỉ khi **bảng đếm chữ cái giống nhau**; các cửa sổ liền kề chỉ khác nhau **một chữ vào, một chữ ra** nên cập nhật bảng đếm trong `O(1)`.
5. Tránh các lỗi kinh điển: dùng `set` thay vì đếm (mất số lần xuất hiện), bỏ sót cửa sổ cuối (`range` lệch 1), xóa nhầm ký tự khi trượt, cập nhật `diff` sai, và nhầm chuỗi con với dãy con.

## 2. Đề bài & ví dụ

Cho hai chuỗi `s1` và `s2`. Trả về `true` nếu `s2` chứa một **hoán vị** của `s1`, ngược lại trả về `false`.

Nói cách khác: trả về `true` nếu **một trong các hoán vị của `s1` là chuỗi con (substring)** của `s2`.

**Ví dụ 1**

- Input: `s1 = "ab"`, `s2 = "eidbaooo"`
- Output: `true`
- Giải thích: `s2` chứa một hoán vị của `s1` là `"ba"` (nằm ở vị trí 3..4).

**Ví dụ 2**

- Input: `s1 = "ab"`, `s2 = "eidboaoo"`
- Output: `false`
- Giải thích: `s2` có chứa `a` và `b` nhưng không đứng sát nhau (`b o a`), nên không có hoán vị nào của `"ab"` làm chuỗi con.

**Ràng buộc:**

- `1 <= s1.length, s2.length <= 10^4`
- `s1` và `s2` chỉ gồm các chữ cái tiếng Anh viết thường.

Chú ý: "chuỗi con" yêu cầu các ký tự **liền nhau**. Nếu `s1` dài hơn `s2` thì hiển nhiên đáp án là `false`.

In [1]:
from typing import List
from collections import Counter
from itertools import permutations
import random
import time

examples = [
    ("ab", "eidbaooo", True),
    ("ab", "eidboaoo", False),
]
for s1, s2, expected in examples:
    print(f"s1={s1!r}, s2={s2!r} -> kỳ vọng {expected} | len(s1)={len(s1)}, len(s2)={len(s2)}")

s1='ab', s2='eidbaooo' -> kỳ vọng True | len(s1)=2, len(s2)=8
s1='ab', s2='eidboaoo' -> kỳ vọng False | len(s1)=2, len(s2)=8


## 3. Kiến thức nền

### 3.1. Hoán vị (anagram) của một chuỗi

Hoán vị của `"abc"` là mọi cách sắp xếp lại các chữ: `abc, acb, bac, bca, cab, cba`. Nếu chuỗi có chữ lặp (như `"aab"`) thì số hoán vị **khác nhau** ít hơn `3!`. Điều quan trọng: hai chuỗi là hoán vị của nhau khi và chỉ khi sau khi **sắp xếp chữ cái** chúng giống hệt nhau, hay nói cách khác khi **mỗi chữ cái xuất hiện cùng số lần**.

In [2]:
print(sorted("ba"), sorted("ab"), sorted("ba") == sorted("ab"))
print("".join(sorted("cabbac")))             # các chữ được xếp về thứ tự: aabbcc
print(sorted("aab") == sorted("aba") == sorted("baa"))

# Liệt kê các hoán vị khác nhau của "aab"
perms = sorted(set("".join(p) for p in permutations("aab")))
print(perms, "-> có", len(perms), "hoán vị khác nhau (không phải 3! = 6)")

['a', 'b'] ['a', 'b'] True
aabbcc
True
['aab', 'aba', 'baa'] -> có 3 hoán vị khác nhau (không phải 3! = 6)


### 3.2. Chuỗi con (substring) khác dãy con (subsequence), và cắt chuỗi `s[i:i+m]`

- **Chuỗi con (substring):** các ký tự **liên tiếp** trong chuỗi gốc. `"ba"` là chuỗi con của `"eidbaooo"`.
- **Dãy con (subsequence):** các ký tự giữ nguyên thứ tự nhưng có thể **bỏ qua** ký tự ở giữa. `"ao"` là dãy con của `"eidbaooo"` nhưng `"ab"` thì không (a đứng sau b).

Bài này cần **chuỗi con**. Trong Python, `s[i:i+m]` cắt `m` ký tự bắt đầu từ vị trí `i` (vị trí `i+m` không lấy). Chuỗi dài `n` có đúng `n - m + 1` chuỗi con độ dài `m` (với `m <= n`).

In [3]:
s2 = "eidbaooo"
m = 2
print("s2[3:5] =", s2[3:5])
windows = [s2[i:i + m] for i in range(len(s2) - m + 1)]
print(windows)
print("số cửa sổ:", len(windows), "= n - m + 1 =", len(s2) - m + 1)

# Không phải chuỗi con vì không liên tiếp:
print("'ab' in s2 ?", "ab" in s2, "| 'ba' in s2 ?", "ba" in s2)

s2[3:5] = ba
['ei', 'id', 'db', 'ba', 'ao', 'oo', 'oo']
số cửa sổ: 7 = n - m + 1 = 7
'ab' in s2 ? False | 'ba' in s2 ? True


### 3.3. `collections.Counter`: bảng đếm chữ cái

`Counter(s)` trả về một dict-đặc-biệt cho biết mỗi ký tự xuất hiện bao nhiêu lần. Hai chuỗi là hoán vị của nhau **khi và chỉ khi** `Counter` của chúng bằng nhau. Một lưu ý: tra một khóa không có trong `Counter` cho `0` (không báo lỗi như dict thường).

Từ Python 3.10, so sánh `==` giữa hai `Counter` coi khóa vắng mặt là `0` (tức `Counter(a=1, b=0) == Counter(a=1)` là `True`). Phiên bản cũ thì khác, nên nếu cần chắc chắn hãy dùng mảng đếm (mục 3.4).

In [4]:
c = Counter("eidbaooo")
print(c)
print("c['o'] =", c["o"], "| c['z'] =", c["z"], "(không có -> 0)")
print(Counter("ab") == Counter("ba"))
print(Counter("aab") == Counter("abb"))
print(Counter("aab") == Counter("aba"))

Counter({'o': 3, 'e': 1, 'i': 1, 'd': 1, 'b': 1, 'a': 1})
c['o'] = 3 | c['z'] = 0 (không có -> 0)
True
False
True


### 3.4. Mảng đếm 26 ô và `ord`

Vì chỉ có 26 chữ cái thường, ta có thể dùng list 26 số nguyên thay `Counter`: ô thứ `k` đếm chữ cái thứ `k` (`a = 0, b = 1, ..., z = 25`). `ord('a')` là mã số của `'a'` (97), nên chỉ số của chữ `c` là `ord(c) - ord('a')`. Mảng này nhanh hơn và dễ so sánh: `cnt1 == cnt2` so sánh cả 26 ô một lần.

In [5]:
def count26(s):
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - ord("a")] += 1
    return cnt

print(ord("a"), ord("b"), ord("z"), "| chỉ số của 'd' =", ord("d") - ord("a"))
print(count26("ab"))
print(count26("ba") == count26("ab"), count26("aab") == count26("abb"))

97 98 122 | chỉ số của 'd' = 3
[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
True False


### 3.5. Cửa sổ trượt (sliding window)

**Cửa sổ** là một đoạn liên tiếp `s2[l..r]`. **Trượt** là dịch cửa sổ sang phải một ô: thêm ký tự mới ở phải (`s2[r+1]`) và bỏ ký tự cũ ở trái (`s2[l]`). Nếu ta lưu sẵn bảng đếm của cửa sổ thì trượt chỉ cần `cnt[mới] += 1` và `cnt[cũ] -= 1`, tức `O(1)`; **không cần đếm lại từ đầu** `m` ký tự.

In [6]:
s2 = "eidbaooo"
m = 2
cnt = count26(s2[:m])                      # cửa sổ đầu tiên "ei"
print("cửa sổ", s2[:m], [(chr(97 + k), v) for k, v in enumerate(cnt) if v])
for i in range(m, len(s2)):
    add, drop = s2[i], s2[i - m]
    cnt[ord(add) - 97] += 1
    cnt[ord(drop) - 97] -= 1
    print("cửa sổ", s2[i - m + 1:i + 1], "| thêm", add, "bớt", drop,
          [(chr(97 + k), v) for k, v in enumerate(cnt) if v])
    assert cnt == count26(s2[i - m + 1:i + 1])      # cập nhật dần phải khớp đếm lại từ đầu

cửa sổ ei [('e', 1), ('i', 1)]
cửa sổ id | thêm d bớt e [('d', 1), ('i', 1)]
cửa sổ db | thêm b bớt i [('b', 1), ('d', 1)]
cửa sổ ba | thêm a bớt d [('a', 1), ('b', 1)]
cửa sổ ao | thêm o bớt b [('a', 1), ('o', 1)]
cửa sổ oo | thêm o bớt a [('o', 2)]
cửa sổ oo | thêm o bớt o [('o', 2)]


### 3.6. `itertools.permutations` để làm "oracle"

`permutations("ab")` sinh `('a','b'), ('b','a')`. Với chuỗi ngắn (độ dài ≤ 6), ta có thể sinh hết hoán vị rồi kiểm tra `p in s2`. Cách này **đúng chắc chắn theo định nghĩa** nhưng có `m!` hoán vị nên chỉ dùng cho input nhỏ, làm chuẩn đối chiếu cho các cách thông minh hơn.

In [7]:
def oracle(s1, s2):
    """Đúng theo định nghĩa: sinh mọi hoán vị của s1 rồi tìm trong s2 (chỉ dùng cho s1 ngắn)."""
    return any("".join(p) in s2 for p in set(permutations(s1)))

for s1, s2, expected in examples:
    print(s1, s2, "->", oracle(s1, s2))
    assert oracle(s1, s2) == expected
print("số hoán vị của chuỗi dài m:", [(m, __import__('math').factorial(m)) for m in (3, 5, 8, 10)])

ab eidbaooo -> True
ab eidboaoo -> False
số hoán vị của chuỗi dài m: [(3, 6), (5, 120), (8, 40320), (10, 3628800)]


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `s1 = "ab"`, `s2 = "eidbaooo"`.

`s1` dài `m = 2`, nên mọi hoán vị của `s1` cũng dài 2: chúng chỉ có thể là `"ab"` hoặc `"ba"`. Vậy ta chỉ cần xem **mọi chuỗi con độ dài đúng 2** của `s2`:

| vị trí | cửa sổ | là hoán vị của `"ab"`? |
|---|---|---|
| 0 | `ei` | không |
| 1 | `id` | không |
| 2 | `db` | không (có `d`) |
| 3 | `ba` | **có** |
| 4 | `ao` | không |
| 5 | `oo` | không |
| 6 | `oo` | không |

Có đúng một cửa sổ khớp, nên đáp án `true`.

Hai quan sát then chốt:

1. **Độ dài cửa sổ phải bằng `m = len(s1)`** (hoán vị không đổi độ dài). Ta không cần xét độ dài khác.
2. Cửa sổ khớp **khi và chỉ khi** bảng đếm chữ của cửa sổ = bảng đếm chữ của `s1`.

Hãy kiểm chứng bằng code, in từng cửa sổ và so sánh bảng đếm.

In [8]:
def show_windows(s1, s2):
    m = len(s1)
    target = Counter(s1)
    for i in range(len(s2) - m + 1):
        w = s2[i:i + m]
        print(f"vị trí {i}: {w!r}  đếm {dict(Counter(w))}  khớp: {Counter(w) == target}")

show_windows("ab", "eidbaooo")

vị trí 0: 'ei'  đếm {'e': 1, 'i': 1}  khớp: False
vị trí 1: 'id'  đếm {'i': 1, 'd': 1}  khớp: False
vị trí 2: 'db'  đếm {'d': 1, 'b': 1}  khớp: False
vị trí 3: 'ba'  đếm {'b': 1, 'a': 1}  khớp: True
vị trí 4: 'ao'  đếm {'a': 1, 'o': 1}  khớp: False
vị trí 5: 'oo'  đếm {'o': 2}  khớp: False
vị trí 6: 'oo'  đếm {'o': 2}  khớp: False


In [9]:
show_windows("ab", "eidboaoo")

vị trí 0: 'ei'  đếm {'e': 1, 'i': 1}  khớp: False
vị trí 1: 'id'  đếm {'i': 1, 'd': 1}  khớp: False
vị trí 2: 'db'  đếm {'d': 1, 'b': 1}  khớp: False
vị trí 3: 'bo'  đếm {'b': 1, 'o': 1}  khớp: False
vị trí 4: 'oa'  đếm {'o': 1, 'a': 1}  khớp: False
vị trí 5: 'ao'  đếm {'a': 1, 'o': 1}  khớp: False
vị trí 6: 'oo'  đếm {'o': 2}  khớp: False


Với ví dụ 2 không cửa sổ nào khớp, đáp án `false`. Từ đây có hai hướng cải tiến: giảm chi phí **mỗi lần so sánh** (đếm thay vì sắp xếp), và giảm chi phí **chuẩn bị mỗi cửa sổ** (cập nhật dần thay vì đếm lại).

## 5. Cách 1 — Brute Force: sắp xếp từng cửa sổ

### Ý tưởng

Với mỗi vị trí `i` từ `0` đến `n - m`, lấy cửa sổ `w = s2[i:i+m]`, rồi so sánh `sorted(w) == sorted(s1)`. Ta tính `sorted(s1)` **một lần** ngoài vòng lặp. Hai chuỗi có cùng dạng sắp xếp thì là hoán vị của nhau.

### Code

In [10]:
def check_inclusion_sorted(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    target = sorted(s1)
    for i in range(n - m + 1):
        if sorted(s2[i:i + m]) == target:
            return True
    return False

for s1, s2, expected in examples:
    got = check_inclusion_sorted(s1, s2)
    print(s1, s2, "->", got)
    assert got == expected

ab eidbaooo -> True
ab eidboaoo -> False


### Độ phức tạp và nút thắt

- **Time:** có `n - m + 1` cửa sổ; sắp xếp mỗi cửa sổ tốn `O(m log m)`. Tổng `O(n · m log m)`. Với `n = 10^4`, `m = 5000` đã vào cỡ `10^8` phép toán trở lên.
- **Space:** `O(m)` cho danh sách đã sắp xếp.

Nút thắt: ta **sắp xếp lại từ đầu** mỗi cửa sổ, dù hai cửa sổ liền kề chỉ khác nhau đúng một chữ vào và một chữ ra.

In [11]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

rng = random.Random(1)
def rand_str(n, alpha="abcdefghijklmnopqrstuvwxyz"):
    return "".join(rng.choice(alpha) for _ in range(n))

for m in (100, 200, 400, 800):
    s1 = rand_str(m)
    s2 = rand_str(6000)
    res, dt = timeit_once(check_inclusion_sorted, s1, s2)
    print(f"m = {m:4d}, n = 6000 -> {res}, {dt * 1000:8.1f} ms")

m =  100, n = 6000 -> False,     76.4 ms


m =  200, n = 6000 -> False,    148.3 ms


m =  400, n = 6000 -> False,    255.5 ms


m =  800, n = 6000 -> False,    528.7 ms


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1 (bản chất hoán vị):** hoán vị ⇔ **cùng bảng đếm**. Vì thế thay vì sắp xếp, ta chỉ cần so sánh hai bảng đếm 26 ô.

**Bước đột phá 2 (cửa sổ trượt):** cửa sổ `s2[i+1 .. i+m]` khác cửa sổ `s2[i .. i+m-1]` đúng ở **hai ký tự**: bỏ `s2[i]`, thêm `s2[i+m]`. Cập nhật bảng đếm chỉ mất `O(1)`.

**Bước đột phá 3 (chỉ đếm "số ô lệch"):** nếu ta đồng thời giữ biến `diff` = số chữ cái mà đếm của cửa sổ **khác** đếm của `s1`, thì cửa sổ khớp khi `diff == 0`, và ta không cần so sánh cả 26 ô mỗi bước. Cập nhật `diff` cũng `O(1)`.

Hãy thử từng bước đột phá bằng số cụ thể: bảng đếm hiệu `cnt = đếm(s1) - đếm(cửa sổ)`; cửa sổ khớp khi mọi ô của `cnt` bằng `0`.

In [12]:
def diff_table(s1, window):
    c = Counter(s1)
    c.subtract(Counter(window))
    return {ch: v for ch, v in sorted(c.items()) if v != 0}

s1 = "ab"
for w in ["ei", "id", "db", "ba"]:
    d = diff_table(s1, w)
    print(f"cửa sổ {w!r}: ô lệch = {d} -> diff = {len(d)}", "-> KHỚP" if not d else "")

cửa sổ 'ei': ô lệch = {'a': 1, 'b': 1, 'e': -1, 'i': -1} -> diff = 4 
cửa sổ 'id': ô lệch = {'a': 1, 'b': 1, 'd': -1, 'i': -1} -> diff = 4 
cửa sổ 'db': ô lệch = {'a': 1, 'd': -1} -> diff = 2 
cửa sổ 'ba': ô lệch = {} -> diff = 0 -> KHỚP


Cột `diff` (4, 4, 2, 0) lần lượt: `ei` lệch 4 chữ (`a, b, e, i`), `id` lệch 4 (`a, b, i, d`), `db` lệch 2 (`a, d`), `ba` lệch 0. Khi `diff` về `0` ta kết luận `true`.

## 7. Các cách tiếp theo

Tổng cộng 5 cách, khác nhau thật về ý tưởng:

1. Sắp xếp từng cửa sổ (đã làm).
2. **Đếm lại từng cửa sổ bằng `Counter`** (bớt `log`, vẫn đếm lại từ đầu).
3. **Cửa sổ trượt cố định, so sánh hai mảng đếm** (cập nhật dần, so sánh 26 ô).
4. **Cửa sổ trượt cố định với biến `diff`** (cập nhật dần, so sánh `O(1)`).
5. **Cửa sổ co giãn hai con trỏ** (không cố định độ dài; co từ trái khi dư chữ).

### 7.1. Cách 2 — Đếm lại từng cửa sổ bằng `Counter`

Giải thích: tính `Counter(s1)` một lần. Với mỗi cửa sổ, tính `Counter(window)` rồi so sánh `==`. Việc đếm `m` ký tự tốn `O(m)`, rẻ hơn sắp xếp `O(m log m)`, nhưng ta vẫn đếm lại từ đầu mỗi cửa sổ nên tổng `O(n · m)`.

In [13]:
def check_inclusion_counter(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    target = Counter(s1)
    for i in range(n - m + 1):
        if Counter(s2[i:i + m]) == target:
            return True
    return False

for s1, s2, expected in examples:
    got = check_inclusion_counter(s1, s2)
    print(s1, s2, "->", got)
    assert got == expected

ab eidbaooo -> True
ab eidboaoo -> False


In [14]:
class Solution2:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        m, n = len(s1), len(s2)
        if m > n:
            return False
        target = Counter(s1)
        for i in range(n - m + 1):
            if Counter(s2[i:i + m]) == target:
                return True
        return False

for s1, s2, expected in examples:
    assert Solution2().checkInclusion(s1, s2) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp Cách 2:** Time `O(n · m)`; Space `O(1)` nếu coi bảng chữ cái cố định 26 (mỗi `Counter` có tối đa 26 khóa).

### 7.2. Cách 3 — Cửa sổ trượt cố định, so sánh hai mảng đếm

Giải thích từng đoạn:

1. Nếu `m > n` trả `False`.
2. `cnt1 = đếm(s1)`, `cnt2 = đếm(s2[:m])` (cửa sổ đầu tiên). Nếu `cnt1 == cnt2` trả `True`.
3. Với `i` từ `m` đến `n - 1`: thêm `s2[i]` vào (`cnt2[...] += 1`), bỏ `s2[i - m]` ra (`cnt2[...] -= 1`). Sau bước này cửa sổ là `s2[i-m+1 .. i]`. Nếu `cnt1 == cnt2` trả `True`.
4. Hết vòng lặp trả `False`.

So sánh `cnt1 == cnt2` duyệt tối đa 26 ô, nên mỗi bước `O(26)`.

In [15]:
def check_inclusion_arrays(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    cnt1 = [0] * 26
    cnt2 = [0] * 26
    for k in range(m):
        cnt1[ord(s1[k]) - 97] += 1
        cnt2[ord(s2[k]) - 97] += 1          # cửa sổ đầu: s2[0..m-1]
    if cnt1 == cnt2:
        return True
    for i in range(m, n):
        cnt2[ord(s2[i]) - 97] += 1          # thêm ký tự mới ở phải
        cnt2[ord(s2[i - m]) - 97] -= 1      # bỏ ký tự cũ ở trái
        if cnt1 == cnt2:
            return True
    return False

for s1, s2, expected in examples:
    got = check_inclusion_arrays(s1, s2)
    print(s1, s2, "->", got)
    assert got == expected

ab eidbaooo -> True
ab eidboaoo -> False


In [16]:
class Solution3:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        m, n = len(s1), len(s2)
        if m > n:
            return False
        cnt1 = [0] * 26
        cnt2 = [0] * 26
        for k in range(m):
            cnt1[ord(s1[k]) - 97] += 1
            cnt2[ord(s2[k]) - 97] += 1
        if cnt1 == cnt2:
            return True
        for i in range(m, n):
            cnt2[ord(s2[i]) - 97] += 1
            cnt2[ord(s2[i - m]) - 97] -= 1
            if cnt1 == cnt2:
                return True
        return False

for s1, s2, expected in examples:
    assert Solution3().checkInclusion(s1, s2) == expected
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n · 26)` = `O(n)` (so sánh 26 ô mỗi cửa sổ); Space `O(26)` = `O(1)`.

### 7.3. Cách 4 — Cửa sổ trượt cố định với biến `diff` (lời giải nộp)

Giải thích từng đoạn:

1. Dùng **một** mảng `cnt` 26 ô duy nhất: `cnt[c] = (số lần c trong s1) - (số lần c trong cửa sổ)`. Cửa sổ khớp khi **mọi ô bằng 0**.
2. Biến `diff` = số ô **khác 0** trong `cnt`. Cửa sổ khớp khi `diff == 0`.
3. Hàm cập nhật `change(c, delta)`: trước khi đổi, nếu `cnt[c] == 0` thì ô này sắp thành khác 0 → `diff += 1`; sau khi đổi, nếu `cnt[c] == 0` thì ô vừa trở về 0 → `diff -= 1`. Vì `delta = ±1`, hai điều kiện không bao giờ cùng đúng.
4. Khởi tạo: cộng `s1` (mỗi chữ `cnt[c] += 1`), trừ cửa sổ đầu `s2[:m]` (mỗi chữ `cnt[c] -= 1`), cập nhật `diff` qua `change`.
5. Trượt: thêm `s2[i]` tức `cnt[s2[i]] -= 1`; bỏ `s2[i-m]` tức `cnt[s2[i-m]] += 1`. Nếu `diff == 0` trả `True`.

Nhờ `diff`, mỗi bước chỉ `O(1)`, không phải so sánh 26 ô.

In [17]:
def check_inclusion_diff(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    cnt = [0] * 26
    diff = 0

    def change(ch, delta):
        nonlocal diff
        k = ord(ch) - 97
        if cnt[k] == 0:
            diff += 1                     # 0 -> khác 0
        cnt[k] += delta
        if cnt[k] == 0:
            diff -= 1                     # khác 0 -> 0

    for k in range(m):
        change(s1[k], +1)                 # s1 làm cnt dương
        change(s2[k], -1)                 # cửa sổ đầu làm cnt âm
    if diff == 0:
        return True
    for i in range(m, n):
        change(s2[i], -1)                 # chữ mới vào
        change(s2[i - m], +1)             # chữ cũ ra
        if diff == 0:
            return True
    return False

for s1, s2, expected in examples:
    got = check_inclusion_diff(s1, s2)
    print(s1, s2, "->", got)
    assert got == expected

ab eidbaooo -> True
ab eidboaoo -> False


In [18]:
class Solution4:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        m, n = len(s1), len(s2)
        if m > n:
            return False
        cnt = [0] * 26
        diff = 0

        def change(ch: str, delta: int) -> None:
            nonlocal diff
            k = ord(ch) - 97
            if cnt[k] == 0:
                diff += 1
            cnt[k] += delta
            if cnt[k] == 0:
                diff -= 1

        for k in range(m):
            change(s1[k], 1)
            change(s2[k], -1)
        if diff == 0:
            return True
        for i in range(m, n):
            change(s2[i], -1)
            change(s2[i - m], 1)
            if diff == 0:
                return True
        return False

for s1, s2, expected in examples:
    assert Solution4().checkInclusion(s1, s2) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp Cách 4:** Time `O(m + n)` (mỗi ký tự được xử lý hằng số lần); Space `O(26)` = `O(1)`.

### 7.4. Cách 5 — Cửa sổ co giãn hai con trỏ (`l`, `r`)

Cách này khác hẳn: **độ dài cửa sổ không cố định**. Ta giữ bảng `need[c]` = số lần chữ `c` **còn được phép** có thêm trong cửa sổ (bắt đầu bằng số lần `c` trong `s1`). Quy tắc:

- Mỗi bước, mở rộng bên phải: nhận `c = s2[r]`, đặt `need[c] -= 1`.
- Nếu `need[c] < 0` (cửa sổ có **dư** chữ `c` so với `s1`), thu hẹp bên trái: lặp `need[s2[l]] += 1; l += 1` cho tới khi `need[c] >= 0` trở lại.
- Sau khi cân bằng, mọi ô `need` đều `>= 0`, tức cửa sổ là **tập con** của `s1` (không chữ nào dư). Nếu độ dài cửa sổ `r - l + 1 == m` thì cửa sổ dùng đúng `m` chữ trong đó không chữ nào vượt quá `s1`, nên **bằng đúng** `s1` về bảng đếm → `True`.

Tại sao vòng `while` luôn dừng? Mỗi lần co ta bỏ một ký tự; trong trường hợp xấu, ta bỏ tới hết cửa sổ (`l = r + 1`) khi `need[c]` quay về `0` hoặc dương. Mỗi con trỏ chỉ đi tới, tổng `O(n)`.

In [19]:
def check_inclusion_two_pointers(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    need = [0] * 26
    for ch in s1:
        need[ord(ch) - 97] += 1
    l = 0
    for r in range(n):
        c = ord(s2[r]) - 97
        need[c] -= 1
        while need[c] < 0:                # dư chữ c: co bên trái
            need[ord(s2[l]) - 97] += 1
            l += 1
        if r - l + 1 == m:
            return True
    return False

for s1, s2, expected in examples:
    got = check_inclusion_two_pointers(s1, s2)
    print(s1, s2, "->", got)
    assert got == expected

ab eidbaooo -> True
ab eidboaoo -> False


In [20]:
class Solution5:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        m, n = len(s1), len(s2)
        if m > n:
            return False
        need = [0] * 26
        for ch in s1:
            need[ord(ch) - 97] += 1
        l = 0
        for r in range(n):
            c = ord(s2[r]) - 97
            need[c] -= 1
            while need[c] < 0:
                need[ord(s2[l]) - 97] += 1
                l += 1
            if r - l + 1 == m:
                return True
        return False

for s1, s2, expected in examples:
    assert Solution5().checkInclusion(s1, s2) == expected
print("Solution5 OK")

Solution5 OK


**Độ phức tạp Cách 5:** Time `O(n)` (mỗi ký tự vào cửa sổ đúng một lần và ra tối đa một lần); Space `O(26)` = `O(1)`.

Bảng so sánh năm cách ở mục 16.

## 8. TRACE từng bước

Trace Cách 4 (biến `diff`) trên ví dụ 1: `s1 = "ab"`, `s2 = "eidbaooo"`, `m = 2`.

`cnt[c] = đếm trong s1 - đếm trong cửa sổ`; `diff` = số ô khác 0.

| bước | chữ vào | chữ ra | cửa sổ | ô khác 0 (cnt) | diff | kết luận |
|---|---|---|---|---|---|---|
| khởi tạo | | | `ei` | a=1, b=1, e=-1, i=-1 | 4 | chưa |
| 1 | d | e | `id` | a=1, b=1, i=-1, d=-1 | 4 | chưa |
| 2 | b | i | `db` | a=1, d=-1 | 2 | chưa |
| 3 | a | d | `ba` | (không có) | 0 | **True** |

Hàm dưới in đúng các bước ấy.

In [21]:
def trace_diff(s1, s2):
    m, n = len(s1), len(s2)
    print(f"s1={s1!r}, s2={s2!r}, m={m}, n={n}")
    if m > n:
        print("s1 dài hơn s2 -> False")
        return False
    cnt = [0] * 26
    diff = 0

    def change(ch, delta):
        nonlocal diff
        k = ord(ch) - 97
        if cnt[k] == 0:
            diff += 1
        cnt[k] += delta
        if cnt[k] == 0:
            diff -= 1

    def nonzero():
        return {chr(97 + k): v for k, v in enumerate(cnt) if v}

    for k in range(m):
        change(s1[k], +1)
        change(s2[k], -1)
    print(f"khởi tạo: cửa sổ {s2[:m]!r} | cnt={nonzero()} | diff={diff}")
    if diff == 0:
        print("-> True")
        return True
    for i in range(m, n):
        change(s2[i], -1)
        change(s2[i - m], +1)
        print(f"i={i}: vào {s2[i]!r}, ra {s2[i - m]!r} | cửa sổ {s2[i - m + 1:i + 1]!r} | cnt={nonzero()} | diff={diff}")
        if diff == 0:
            print("-> True")
            return True
    print("-> False")
    return False

assert trace_diff("ab", "eidbaooo") is True

s1='ab', s2='eidbaooo', m=2, n=8
khởi tạo: cửa sổ 'ei' | cnt={'a': 1, 'b': 1, 'e': -1, 'i': -1} | diff=4
i=2: vào 'd', ra 'e' | cửa sổ 'id' | cnt={'a': 1, 'b': 1, 'd': -1, 'i': -1} | diff=4
i=3: vào 'b', ra 'i' | cửa sổ 'db' | cnt={'a': 1, 'd': -1} | diff=2
i=4: vào 'a', ra 'd' | cửa sổ 'ba' | cnt={} | diff=0
-> True


In [22]:
assert trace_diff("ab", "eidboaoo") is False

s1='ab', s2='eidboaoo', m=2, n=8
khởi tạo: cửa sổ 'ei' | cnt={'a': 1, 'b': 1, 'e': -1, 'i': -1} | diff=4
i=2: vào 'd', ra 'e' | cửa sổ 'id' | cnt={'a': 1, 'b': 1, 'd': -1, 'i': -1} | diff=4
i=3: vào 'b', ra 'i' | cửa sổ 'db' | cnt={'a': 1, 'd': -1} | diff=2
i=4: vào 'o', ra 'd' | cửa sổ 'bo' | cnt={'a': 1, 'o': -1} | diff=2
i=5: vào 'a', ra 'b' | cửa sổ 'oa' | cnt={'b': 1, 'o': -1} | diff=2
i=6: vào 'o', ra 'o' | cửa sổ 'ao' | cnt={'b': 1, 'o': -1} | diff=2
i=7: vào 'o', ra 'a' | cửa sổ 'oo' | cnt={'a': 1, 'b': 1, 'o': -2} | diff=3
-> False


Trace Cách 5 (hai con trỏ) trên ví dụ 1: `need` ban đầu `a=1, b=1`.

| r | `s2[r]` | need sau khi trừ | co trái? | `l` sau đó | cửa sổ `s2[l..r]` | độ dài | kết luận |
|---|---|---|---|---|---|---|---|
| 0 | e | e=-1 | có (bỏ `e`, need[e]=0) | 1 | rỗng | 0 | chưa |
| 1 | i | i=-1 | có (bỏ `i`) | 2 | rỗng | 0 | chưa |
| 2 | d | d=-1 | có (bỏ `d`) | 3 | rỗng | 0 | chưa |
| 3 | b | b=0 | không | 3 | `b` | 1 | chưa |
| 4 | a | a=0 | không | 3 | `ba` | 2 = m | **True** |

In [23]:
def trace_two_pointers(s1, s2):
    m, n = len(s1), len(s2)
    need = [0] * 26
    for ch in s1:
        need[ord(ch) - 97] += 1
    show = lambda: {chr(97 + k): v for k, v in enumerate(need) if v}
    print(f"s1={s1!r}, s2={s2!r}, need ban đầu={show()}")
    l = 0
    for r in range(n):
        c = ord(s2[r]) - 97
        need[c] -= 1
        shrunk = []
        while need[c] < 0:
            shrunk.append(s2[l])
            need[ord(s2[l]) - 97] += 1
            l += 1
        print(f"r={r}: nhận {s2[r]!r}, co trái bỏ {shrunk} -> l={l}, cửa sổ {s2[l:r + 1]!r} (dài {r - l + 1}), need={show()}")
        if r - l + 1 == m:
            print("-> True")
            return True
    print("-> False")
    return False

assert trace_two_pointers("ab", "eidbaooo") is True

s1='ab', s2='eidbaooo', need ban đầu={'a': 1, 'b': 1}
r=0: nhận 'e', co trái bỏ ['e'] -> l=1, cửa sổ '' (dài 0), need={'a': 1, 'b': 1}
r=1: nhận 'i', co trái bỏ ['i'] -> l=2, cửa sổ '' (dài 0), need={'a': 1, 'b': 1}
r=2: nhận 'd', co trái bỏ ['d'] -> l=3, cửa sổ '' (dài 0), need={'a': 1, 'b': 1}
r=3: nhận 'b', co trái bỏ [] -> l=3, cửa sổ 'b' (dài 1), need={'a': 1}
r=4: nhận 'a', co trái bỏ [] -> l=3, cửa sổ 'ba' (dài 2), need={}
-> True


Một trace có chữ lặp để thấy vòng `while` co **nhiều lần**: `s1 = "aab"`, `s2 = "aaaab"`. Ở `r = 2` ta có ba chữ `a` nhưng `s1` chỉ cho phép hai nên phải bỏ một `a` bên trái.

In [24]:
assert trace_two_pointers("aab", "aaaab") is True

s1='aab', s2='aaaab', need ban đầu={'a': 2, 'b': 1}
r=0: nhận 'a', co trái bỏ [] -> l=0, cửa sổ 'a' (dài 1), need={'a': 1, 'b': 1}
r=1: nhận 'a', co trái bỏ [] -> l=0, cửa sổ 'aa' (dài 2), need={'b': 1}
r=2: nhận 'a', co trái bỏ ['a'] -> l=1, cửa sổ 'aa' (dài 2), need={'b': 1}
r=3: nhận 'a', co trái bỏ ['a'] -> l=2, cửa sổ 'aa' (dài 2), need={'b': 1}
r=4: nhận 'b', co trái bỏ [] -> l=2, cửa sổ 'aab' (dài 3), need={}
-> True


## 9. Code nộp LeetCode

Ta chọn **Cách 4 (cửa sổ cố định + `diff`)**: `O(n)`, bộ nhớ hằng số, logic rõ ràng. (Cách 5 cũng nộp được.)

In [25]:
class Solution:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        m, n = len(s1), len(s2)
        if m > n:
            return False
        cnt = [0] * 26
        diff = 0

        def change(ch: str, delta: int) -> None:
            nonlocal diff
            k = ord(ch) - 97
            if cnt[k] == 0:
                diff += 1
            cnt[k] += delta
            if cnt[k] == 0:
                diff -= 1

        for k in range(m):
            change(s1[k], 1)
            change(s2[k], -1)
        if diff == 0:
            return True
        for i in range(m, n):
            change(s2[i], -1)
            change(s2[i - m], 1)
            if diff == 0:
                return True
        return False


sol = Solution()
for s1, s2, expected in examples:
    got = sol.checkInclusion(s1, s2)
    print(s1, s2, "->", got)
    assert got == expected

ab eidbaooo -> True
ab eidboaoo -> False


## 10. Vì sao đúng

**Khẳng định 1 (hoán vị ⇔ cùng bảng đếm).** Nếu `t` là hoán vị của `s1` thì chỉ khác thứ tự nên mỗi chữ xuất hiện cùng số lần. Ngược lại, nếu hai chuỗi có cùng bảng đếm thì ta có thể xếp các chữ của `s1` theo đúng thứ tự trong `t` (vì có đủ từng loại chữ), nên `t` là hoán vị của `s1`. Do đó "tồn tại hoán vị của `s1` là chuỗi con của `s2`" ⇔ "tồn tại chuỗi con của `s2` có bảng đếm bằng bảng đếm của `s1`". Mọi hoán vị đều dài đúng `m`, nên chỉ cần xét các chuỗi con **độ dài `m`**, tức `n - m + 1` cửa sổ `s2[i..i+m-1]`, `i = 0..n-m`.

**Khẳng định 2 (cập nhật trượt đúng).** Cửa sổ ở bước `i` (là `s2[i-m+1..i]`) có bảng đếm bằng bảng đếm cửa sổ trước cộng `s2[i]` trừ `s2[i-m]`. Cửa sổ đầu tiên `s2[0..m-1]` được khởi tạo trực tiếp. Quy nạp theo `i`, bảng đếm luôn đúng.

**Khẳng định 3 (`diff` đúng).** Bất biến: `diff` = số `k` có `cnt[k] != 0`. Hàm `change` chỉ đụng tới đúng một ô `k` mỗi lần: nếu ô từ `0` sang khác `0` thì tăng `diff` đúng 1; nếu từ khác `0` về `0` thì giảm đúng 1; nếu khác `0` sang khác `0` (như `1 -> 2` hay `2 -> 1`) thì `diff` không đổi, mà quy tắc của ta cũng không đổi `diff` vì hai điều kiện `cnt[k] == 0` (trước và sau) đều sai. Với `delta = ±1`, ô không thể ở `0` cả trước lẫn sau. Vậy bất biến được giữ. Khi `diff == 0` mọi `cnt[k] = 0`, tức bảng đếm cửa sổ = bảng đếm `s1`.

**Khẳng định 4 (hai con trỏ đúng).** Bất biến sau mỗi bước `r`: mọi `need[c] >= 0`, tức cửa sổ `s2[l..r]` không chứa chữ nào nhiều hơn trong `s1`. Sau khi nhận `s2[r]`, chỉ ô `c = s2[r]` có thể âm; vòng `while` co `l` cho tới khi `need[c] >= 0`; việc co chỉ cộng lại các ô nên không làm ô nào âm. Nếu độ dài cửa sổ `= m` và mọi `need[c] >= 0` thì tổng `need` ban đầu = `m` đã bị trừ đúng `m` lần, mỗi ô vẫn `>= 0` nên **tất cả đều bằng 0**, tức cửa sổ có đúng bảng đếm của `s1`. Ngược lại nếu tồn tại cửa sổ khớp `s2[a..a+m-1]`: ta chứng minh khi `r = a+m-1` thì `l <= a` (con trỏ trái chưa vượt quá `a`, vì mọi chuỗi con của một cửa sổ khớp cũng thỏa "không dư chữ", nên vòng `while` không bao giờ co quá `a`); mà cửa sổ hiện tại là chuỗi con **không dư** kết thúc ở `r` và chứa `s2[a..r]` nên có độ dài `>= m`, mà tối đa là `m`, vậy đúng bằng `m` → trả `True`.

**Trường hợp `m > n`:** không thể có chuỗi con dài `m` nên trả `False` ngay.

## 11. Time / Space Complexity

- **Cách 1 (sắp xếp mỗi cửa sổ):** Time `O(n · m log m)`; Space `O(m)`.
- **Cách 2 (`Counter` mỗi cửa sổ):** Time `O(n · m)`; Space `O(1)` (26 khóa).
- **Cách 3 (hai mảng đếm):** Time `O(26 · n)`; Space `O(1)`.
- **Cách 4 (`diff`, nộp):** Time `O(n + m)`; Space `O(1)`.
- **Cách 5 (hai con trỏ):** Time `O(n + m)`; Space `O(1)`.

Hãy đo với dữ liệu lớn: `n = 10^4`, `m` thay đổi. Ta chọn `s2` ngẫu nhiên và `s1` ngẫu nhiên nên gần như chắc chắn không có hoán vị (trường hợp xấu: phải duyệt hết).

In [26]:
methods = [
    ("sorted", check_inclusion_sorted),
    ("Counter", check_inclusion_counter),
    ("arrays", check_inclusion_arrays),
    ("diff", check_inclusion_diff),
    ("2-ptr", check_inclusion_two_pointers),
]
rng = random.Random(11)
n = 10 ** 4
s2_big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(n))
print("n = 10000; thời gian (ms) cho từng cách")
print(f"{'m':>6} | " + " | ".join(f"{name:>8}" for name, _ in methods))
for m in (10, 100, 1000, 3000):
    s1_big = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(m))
    row = []
    answers = set()
    for name, fn in methods:
        res, dt = timeit_once(fn, s1_big, s2_big)
        answers.add(res)
        row.append(f"{dt * 1000:8.1f}")
    assert len(answers) == 1
    print(f"{m:>6} | " + " | ".join(row))

n = 10000; thời gian (ms) cho từng cách
     m |   sorted |  Counter |   arrays |     diff |    2-ptr
    10 |      8.7 |     18.4 |      1.2 |      2.8 |      1.8


   100 |    113.5 |     45.0 |      1.2 |      2.6 |      1.8


  1000 |   1194.7 |    238.3 |      1.3 |      2.5 |      1.7


  3000 |   2920.9 |    576.0 |      2.0 |      4.1 |      2.9


Quan sát: `sorted` và `Counter` tăng rõ rệt theo `m`, còn ba cách còn lại gần như không đổi. Với ràng buộc `10^4` của đề, ba cách tuyến tính đều chạy trong vài mili-giây đến vài chục mili-giây.

## 12. Bộ test

Ta kiểm tra bốn lớp: ví dụ đề, edge case, phản ví dụ cho các cách "gần đúng", và đối chiếu có hệ thống/ngẫu nhiên (có seed) với oracle `itertools.permutations`.

In [27]:
all_methods = {
    "sorted": check_inclusion_sorted,
    "Counter": check_inclusion_counter,
    "arrays": check_inclusion_arrays,
    "diff": check_inclusion_diff,
    "2-ptr": check_inclusion_two_pointers,
    "submit": lambda a, b: Solution().checkInclusion(a, b),
    "Sol2": lambda a, b: Solution2().checkInclusion(a, b),
    "Sol3": lambda a, b: Solution3().checkInclusion(a, b),
    "Sol4": lambda a, b: Solution4().checkInclusion(a, b),
    "Sol5": lambda a, b: Solution5().checkInclusion(a, b),
}

def run_all(s1, s2, expected=None):
    exp = oracle(s1, s2) if expected is None else expected
    for name, fn in all_methods.items():
        got = fn(s1, s2)
        assert got == exp, (name, s1, s2, got, exp)
    return exp

for s1, s2, expected in examples:
    run_all(s1, s2, expected)
print("ví dụ đề: OK")

ví dụ đề: OK


In [28]:
edge = [
    ("a", "a", True),                 # nhỏ nhất, khớp
    ("a", "b", False),                # nhỏ nhất, không khớp
    ("ab", "a", False),               # s1 dài hơn s2
    ("abc", "abc", True),             # s1 == s2
    ("abc", "cba", True),             # s2 là hoán vị đúng
    ("abc", "abd", False),
    ("ab", "ba", True),
    ("ab", "xba", True),              # cửa sổ khớp nằm ở CUỐI s2 (bắt lỗi range lệch 1)
    ("ab", "abx", True),              # cửa sổ khớp nằm ở ĐẦU s2
    ("aab", "abbaa", True),           # chữ lặp: "baa" khớp
    ("aab", "abab", True),            # cửa sổ "aba" có a=2, b=1 nên khớp
    ("aab", "abbb", False),           # thiếu một chữ a
    ("z", "abcz", True),              # chữ cuối bảng chữ cái
    ("aaa", "aabaa", False),          # lặp nhiều, bị chặn bởi b
    ("aaa", "baaab", True),
]
for s1, s2, expected in edge:
    assert run_all(s1, s2, expected) == expected == oracle(s1, s2)
    print(f"{s1!r:8} {s2!r:10} -> {expected}")
print("edge case: tất cả", len(edge), "trường hợp đúng")

'a'      'a'        -> True
'a'      'b'        -> False
'ab'     'a'        -> False
'abc'    'abc'      -> True
'abc'    'cba'      -> True
'abc'    'abd'      -> False
'ab'     'ba'       -> True
'ab'     'xba'      -> True
'ab'     'abx'      -> True
'aab'    'abbaa'    -> True
'aab'    'abab'     -> True
'aab'    'abbb'     -> False
'z'      'abcz'     -> True
'aaa'    'aabaa'    -> False
'aaa'    'baaab'    -> True
edge case: tất cả 15 trường hợp đúng


Phản ví dụ cho các cách "gần đúng" người mới hay nghĩ ra:

- *So sánh bằng `set`* (`set(window) == set(s1)`): sai vì mất số lần xuất hiện. `s1 = "aab"`, `s2 = "abb"`: `set` giống nhau nhưng không phải hoán vị.
- *Chỉ kiểm tra `s1` có phải dãy con của `s2` hay các chữ của `s1` đều có trong `s2`*: `s1 = "ab"`, `s2 = "eidboaoo"` có đủ `a, b` nhưng đáp án `False`.
- *So sánh tổng mã ký tự* (`sum(map(ord, window)) == sum(map(ord, s1))`): sai vì khác chữ vẫn có thể cùng tổng, ví dụ `"ad"` và `"bc"` (`97+100 = 98+99`).

In [29]:
def wrong_set_compare(s1, s2):
    m = len(s1)
    return any(set(s2[i:i + m]) == set(s1) for i in range(len(s2) - m + 1))

def wrong_sum_compare(s1, s2):
    m = len(s1)
    t = sum(map(ord, s1))
    return any(sum(map(ord, s2[i:i + m])) == t for i in range(len(s2) - m + 1))

def wrong_letters_exist(s1, s2):
    return all(c in s2 for c in s1)

print("set:   ('aab','abb')          đúng =", oracle("aab", "abb"), "| sai =", wrong_set_compare("aab", "abb"))
print("sum:   ('ad','bc')            đúng =", oracle("ad", "bc"), "| sai =", wrong_sum_compare("ad", "bc"))
print("exist: ('ab','eidboaoo')      đúng =", oracle("ab", "eidboaoo"), "| sai =", wrong_letters_exist("ab", "eidboaoo"))
assert wrong_set_compare("aab", "abb") != oracle("aab", "abb")
assert wrong_sum_compare("ad", "bc") != oracle("ad", "bc")
assert wrong_letters_exist("ab", "eidboaoo") != oracle("ab", "eidboaoo")

set:   ('aab','abb')          đúng = False | sai = True
sum:   ('ad','bc')            đúng = False | sai = True
exist: ('ab','eidboaoo')      đúng = False | sai = True


In [30]:
# Đối chiếu có hệ thống: MỌI s1 độ dài 1..3 và MỌI s2 độ dài 1..7 trên bảng chữ "ab" (và "abc" cho độ dài nhỏ hơn)
from itertools import product

total = 0
for alpha, max1, max2 in (("ab", 3, 7), ("abc", 3, 5)):
    for L1 in range(1, max1 + 1):
        for t1 in product(alpha, repeat=L1):
            s1 = "".join(t1)
            for L2 in range(1, max2 + 1):
                for t2 in product(alpha, repeat=L2):
                    run_all(s1, "".join(t2))
                    total += 1
print("vét cạn", total, "cặp (s1, s2): mọi cách đều khớp oracle")

vét cạn 17713 cặp (s1, s2): mọi cách đều khớp oracle


In [31]:
# Ngẫu nhiên có seed, đúng ràng buộc (chữ thường); s1 ngắn để oracle chạy nổi (<= 6)
rng = random.Random(2025)
true_count = 0
for t in range(1500):
    alpha = rng.choice(["ab", "abc", "abcd", "abcdefghijklmnopqrstuvwxyz"])
    m = rng.randint(1, 6)
    n = rng.randint(1, 25)
    s1 = "".join(rng.choice(alpha) for _ in range(m))
    if rng.random() < 0.5 and n >= m:
        # chủ động chèn một hoán vị của s1 để có nhiều đáp án True
        chars = list(s1)
        rng.shuffle(chars)
        pos = rng.randint(0, n - m)
        base = [rng.choice(alpha) for _ in range(n)]
        base[pos:pos + m] = chars
        s2 = "".join(base)
    else:
        s2 = "".join(rng.choice(alpha) for _ in range(n))
    true_count += run_all(s1, s2)
print("1500 test ngẫu nhiên: OK, trong đó", true_count, "test có đáp án True")

1500 test ngẫu nhiên: OK, trong đó 1019 test có đáp án True


In [32]:
# Test lớn đúng ràng buộc (chỉ các cách nhanh): đáp án biết trước
fast = ["arrays", "diff", "2-ptr", "submit", "Sol3", "Sol4", "Sol5"]
n = 10 ** 4

# 1) s1 = s2 chỉ có 'a' -> True; s1 có thêm một 'b' -> False
s1 = "a" * 5000
s2 = "a" * n
assert all(all_methods[k](s1, s2) for k in fast)
s1 = "a" * 4999 + "b"
assert not any(all_methods[k](s1, s2) for k in fast)

# 2) hoán vị nằm ở đúng cuối s2
rng = random.Random(3)
s1 = "".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(3000))
perm = list(s1)
rng.shuffle(perm)
s2 = "".join("z" if rng.random() < 0.5 else "y" for _ in range(n - 3000)) + "".join(perm)
assert len(s2) == n
assert all(all_methods[k](s1, s2) for k in fast)

# 3) s1 dài nhất bằng s2, không khớp
s1 = "a" * (n - 1) + "b"
s2 = "b" + "a" * (n - 1)
assert all(all_methods[k](s1, s2) for k in fast)      # hoán vị đúng (s2 là hoán vị của s1)
s2 = "c" + "a" * (n - 1)
assert not any(all_methods[k](s1, s2) for k in fast)
print("test lớn n = 10^4: OK")

test lớn n = 10^4: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `set` thay vì đếm (mất số lần xuất hiện)

Code sai: `set(window) == set(s1)`. Với `s1 = "aab"` và cửa sổ `"abb"`, hai `set` đều là `{a, b}` nhưng `"abb"` không phải hoán vị của `"aab"` (số `a` và `b` khác). Cần **đếm** (mảng đếm hay `Counter`), không chỉ kiểm tra có mặt.

In [33]:
print(set("aab") == set("abb"), "(sai: tưởng khớp)")
print(Counter("aab") == Counter("abb"), "(đúng: không khớp)")
print("kết quả bài toán ('aab','abb'):", Solution().checkInclusion("aab", "abb"))

True (sai: tưởng khớp)
False (đúng: không khớp)
kết quả bài toán ('aab','abb'): False


### Lỗi 2 — Bỏ sót cửa sổ cuối (`range` lệch 1)

Số cửa sổ độ dài `m` là `n - m + 1`, nên chỉ số bắt đầu chạy `0..n-m` **bao gồm** `n - m`. Viết `range(n - m)` sẽ bỏ cửa sổ cuối. Test `("ab", "xba")` bắt được lỗi này vì cửa sổ khớp là cửa sổ cuối.

In [34]:
def buggy_range(s1, s2):
    m, n = len(s1), len(s2)
    target = sorted(s1)
    for i in range(n - m):                 # BUG: thiếu "+ 1"
        if sorted(s2[i:i + m]) == target:
            return True
    return False

print("('ab','xba')  sai :", buggy_range("ab", "xba"), "| đúng :", Solution().checkInclusion("ab", "xba"))
print("('ab','ba')   sai :", buggy_range("ab", "ba"), "| đúng :", Solution().checkInclusion("ab", "ba"))
assert buggy_range("ab", "xba") != Solution().checkInclusion("ab", "xba")

('ab','xba')  sai : False | đúng : True
('ab','ba')   sai : False | đúng : True


### Lỗi 3 — Xóa nhầm ký tự khi trượt (lệch 1 so với `i - m`)

Khi thêm `s2[i]`, ký tự phải bỏ là `s2[i - m]` (ký tự ngay trước cửa sổ mới). Nếu viết `s2[i - m + 1]`, ta bỏ nhầm ký tự **vẫn còn trong cửa sổ**, bảng đếm trôi dần và kết quả sai.

In [35]:
def buggy_slide(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    cnt1, cnt2 = [0] * 26, [0] * 26
    for k in range(m):
        cnt1[ord(s1[k]) - 97] += 1
        cnt2[ord(s2[k]) - 97] += 1
    if cnt1 == cnt2:
        return True
    for i in range(m, n):
        cnt2[ord(s2[i]) - 97] += 1
        cnt2[ord(s2[i - m + 1]) - 97] -= 1     # BUG: phải là s2[i - m]
        if cnt1 == cnt2:
            return True
    return False

print("('ab','eidbaooo') sai:", buggy_slide("ab", "eidbaooo"), "| đúng:", True)

# Tìm một phản ví dụ bằng cách thử ngẫu nhiên có seed
rng = random.Random(4)
found = None
for _ in range(2000):
    s1 = "".join(rng.choice("ab") for _ in range(rng.randint(1, 3)))
    s2 = "".join(rng.choice("ab") for _ in range(rng.randint(1, 6)))
    if buggy_slide(s1, s2) != oracle(s1, s2):
        found = (s1, s2)
        break
print("phản ví dụ:", found, "| sai =", buggy_slide(*found), "| đúng =", oracle(*found))

('ab','eidbaooo') sai: False | đúng: True
phản ví dụ: ('bb', 'baaaba') | sai = True | đúng = False


### Lỗi 4 — Cập nhật `diff` sai điều kiện

Code sai thường gặp: sau khi đổi `cnt[k]`, viết `if cnt[k] == 0: diff -= 1 else: diff += 1`. Khi ô đi từ `1` lên `2` (khác 0 sang khác 0) thì `diff` bị tăng oan. Quy tắc đúng: nhìn **trước** (có đang `0` không → sắp khác 0 → `+1`) và **sau** (vừa về `0` → `-1`).

In [36]:
def buggy_diff(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    cnt = [0] * 26
    diff = 0

    def change(ch, delta):
        nonlocal diff
        k = ord(ch) - 97
        cnt[k] += delta
        if cnt[k] == 0:
            diff -= 1
        else:
            diff += 1                      # BUG: tăng oan cả khi ô vốn đã khác 0

    for k in range(m):
        change(s1[k], 1)
        change(s2[k], -1)
    if diff == 0:
        return True
    for i in range(m, n):
        change(s2[i], -1)
        change(s2[i - m], 1)
        if diff == 0:
            return True
    return False

rng = random.Random(8)
found = None
for _ in range(2000):
    s1 = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    s2 = "".join(rng.choice("ab") for _ in range(rng.randint(1, 8)))
    if buggy_diff(s1, s2) != oracle(s1, s2):
        found = (s1, s2)
        break
print("phản ví dụ:", found, "| sai =", buggy_diff(*found), "| đúng =", oracle(*found))

phản ví dụ: ('ab', 'bbbaaabb') | sai = False | đúng = True


### Lỗi 5 — Cách 5 dùng `if` thay cho `while` khi co cửa sổ

Khi nhận một chữ làm `need[c]` âm, có thể phải bỏ **nhiều** ký tự bên trái (không chỉ một) mới cân bằng. Dùng `if` chỉ bỏ đúng một ký tự, cửa sổ vẫn dư chữ và bất biến hỏng.

In [37]:
def buggy_if_shrink(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    need = [0] * 26
    for ch in s1:
        need[ord(ch) - 97] += 1
    l = 0
    for r in range(n):
        c = ord(s2[r]) - 97
        need[c] -= 1
        if need[c] < 0:                    # BUG: phải là while
            need[ord(s2[l]) - 97] += 1
            l += 1
        if r - l + 1 == m:
            return True
    return False

rng = random.Random(9)
found = None
for _ in range(5000):
    s1 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 4)))
    s2 = "".join(rng.choice("abc") for _ in range(rng.randint(1, 9)))
    if buggy_if_shrink(s1, s2) != oracle(s1, s2):
        found = (s1, s2)
        break
print("phản ví dụ:", found, "| sai =", buggy_if_shrink(*found), "| đúng =", oracle(*found))

phản ví dụ: ('ba', 'acaccaa') | sai = True | đúng = False


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "chuỗi con liên tiếp thỏa tính chất về **bảng đếm chữ** (hoán vị, anagram) với độ dài cố định" ⇒ **cửa sổ trượt độ dài cố định** + bảng đếm.
- Hoán vị/anagram ⇔ **cùng bảng đếm** ⇔ cùng dạng sắp xếp. Đếm rẻ hơn sắp xếp.
- Trượt cửa sổ: **vào một, ra một**; cập nhật `O(1)`, đừng đếm lại từ đầu.
- Muốn tránh so sánh 26 ô mỗi bước: dùng biến **`diff`** (số ô lệch) hoặc biến "số chữ đã thỏa".
- Cửa sổ **co giãn**: khi vi phạm "dư chữ" thì thu hẹp trái bằng `while`; điều kiện thành công thường là độ dài khớp.
- Luôn xét `len(s1) > len(s2)` ⇒ `False`, và số cửa sổ là `n - m + 1`.
- Dùng oracle `itertools.permutations` với input nhỏ để bắt lỗi lệch 1.
- Các bài cùng họ: **438. Find All Anagrams in a String**, **76. Minimum Window Substring**, **242. Valid Anagram**, **3. Longest Substring Without Repeating Characters**, **1456. Maximum Number of Vowels in a Substring of Given Length**.

## 15. Biến thể

### 15.1. Trả về vị trí bắt đầu đầu tiên (hoặc `-1`)

Chỉ cần thay `return True` bằng chỉ số đầu cửa sổ `i - m + 1`.

In [38]:
def first_permutation_index(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return -1
    cnt = [0] * 26
    diff = 0

    def change(ch, delta):
        nonlocal diff
        k = ord(ch) - 97
        if cnt[k] == 0:
            diff += 1
        cnt[k] += delta
        if cnt[k] == 0:
            diff -= 1

    for k in range(m):
        change(s1[k], 1)
        change(s2[k], -1)
    if diff == 0:
        return 0
    for i in range(m, n):
        change(s2[i], -1)
        change(s2[i - m], 1)
        if diff == 0:
            return i - m + 1
    return -1

assert first_permutation_index("ab", "eidbaooo") == 3
assert first_permutation_index("ab", "eidboaoo") == -1
assert first_permutation_index("abc", "abc") == 0
print(first_permutation_index("ab", "eidbaooo"))

3


### 15.2. Tìm **tất cả** vị trí (bài 438 — Find All Anagrams in a String)

Không trả về ngay khi `diff == 0`, mà ghi lại vị trí rồi tiếp tục trượt.

In [39]:
def find_all_anagrams(s, p):
    m, n = len(p), len(s)
    if m > n:
        return []
    cnt = [0] * 26
    diff = 0
    res = []

    def change(ch, delta):
        nonlocal diff
        k = ord(ch) - 97
        if cnt[k] == 0:
            diff += 1
        cnt[k] += delta
        if cnt[k] == 0:
            diff -= 1

    for k in range(m):
        change(p[k], 1)
        change(s[k], -1)
    if diff == 0:
        res.append(0)
    for i in range(m, n):
        change(s[i], -1)
        change(s[i - m], 1)
        if diff == 0:
            res.append(i - m + 1)
    return res

assert find_all_anagrams("cbaebabacd", "abc") == [0, 6]
assert find_all_anagrams("abab", "ab") == [0, 1, 2]
print(find_all_anagrams("cbaebabacd", "abc"))

[0, 6]


### 15.3. Bảng chữ cái tùy ý (Unicode, chữ hoa, chữ số): dùng `Counter` / `dict`

Mảng 26 ô chỉ hợp với chữ thường. Với ký tự bất kỳ, thay mảng bằng `dict`; biến `diff` vẫn dùng được nếu ta nhớ **xóa khóa** hoặc kiểm tra `== 0` qua `.get`.

In [40]:
def check_inclusion_any_chars(s1, s2):
    m, n = len(s1), len(s2)
    if m > n:
        return False
    cnt = {}
    diff = 0

    def change(ch, delta):
        nonlocal diff
        old = cnt.get(ch, 0)
        if old == 0:
            diff += 1
        cnt[ch] = old + delta
        if cnt[ch] == 0:
            diff -= 1

    for k in range(m):
        change(s1[k], 1)
        change(s2[k], -1)
    if diff == 0:
        return True
    for i in range(m, n):
        change(s2[i], -1)
        change(s2[i - m], 1)
        if diff == 0:
            return True
    return False

assert check_inclusion_any_chars("Ab9", "xx9bAyy") is True
assert check_inclusion_any_chars("Ab9", "xx9bayy") is False       # 'a' khác 'A'
assert check_inclusion_any_chars("đạt", "xxtạđ") is True            # có dấu tiếng Việt
rng = random.Random(6)
for _ in range(300):
    s1 = "".join(rng.choice("abcAB") for _ in range(rng.randint(1, 4)))
    s2 = "".join(rng.choice("abcAB") for _ in range(rng.randint(1, 9)))
    assert check_inclusion_any_chars(s1, s2) == oracle(s1, s2)
print("any chars OK")

any chars OK


### 15.4. Cửa sổ nhỏ nhất chứa đủ các chữ của `s1` (bài 76 — Minimum Window Substring)

Đây là phiên bản **co giãn** nặng hơn: tìm chuỗi con **ngắn nhất** của `s` chứa mọi ký tự của `t` (đủ số lần, thứ tự tùy ý, có thể dư chữ khác). Mở rộng phải cho tới khi đủ, rồi co trái để ngắn nhất có thể; dùng biến `missing` = số ký tự còn thiếu.

In [41]:
def min_window(s, t):
    if not t or len(t) > len(s):
        return ""
    need = Counter(t)
    missing = len(t)
    best = (0, 0)               # [lo, hi) của đáp án; (0,0) nghĩa là chưa có
    best_len = float("inf")
    l = 0
    for r, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        while missing == 0:     # đã đủ: thử co trái
            if r - l + 1 < best_len:
                best_len = r - l + 1
                best = (l, r + 1)
            need[s[l]] += 1
            if need[s[l]] > 0:
                missing += 1
            l += 1
    return s[best[0]:best[1]]

def min_window_brute(s, t):
    best = ""
    for i in range(len(s)):
        for j in range(i + 1, len(s) + 1):
            w = s[i:j]
            if not Counter(t) - Counter(w) and (best == "" or len(w) < len(best)):
                best = w
    return best

assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
assert min_window("a", "aa") == ""
rng = random.Random(12)
for _ in range(500):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    t = "".join(rng.choice("abc") for _ in range(rng.randint(1, 4)))
    got, exp = min_window(s, t), min_window_brute(s, t)
    assert len(got) == len(exp) and (got == "" or not Counter(t) - Counter(got) and got in s), (s, t, got, exp)
print("min_window OK:", min_window("ADOBECODEBANC", "ABC"))

min_window OK: BANC


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp xếp từng cửa sổ | `sorted(window) == sorted(s1)` | `O(n · m log m)` | `O(m)` | Viết nhanh khi `m` nhỏ; baseline |
| 2. `Counter` từng cửa sổ | Đếm lại từ đầu rồi so sánh | `O(n · m)` | `O(1)` | Dễ hiểu; vẫn đếm lại nhiều |
| 3. Hai mảng đếm, trượt | Cập nhật dần, so sánh 26 ô | `O(26 · n)` | `O(1)` | Đơn giản và đủ nhanh |
| 4. `diff` + trượt (nộp) | Theo dõi số ô lệch, so sánh `O(1)` | `O(n + m)` | `O(1)` | Mặc định cho bài anagram cửa sổ cố định |
| 5. Hai con trỏ co giãn | Co trái khi dư chữ, đủ độ dài `m` | `O(n + m)` | `O(1)` | Khi độ dài cửa sổ không cố định (bài 76, 3) |

Cách 4 và 5 đều tuyến tính; cách 4 dễ chứng minh hơn, cách 5 là bước đệm để học các bài cửa sổ co giãn.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của bốn dòng sau, rồi chạy để kiểm tra:

```python
print(ord("d") - ord("a"))
print(sorted("ba") == sorted("ab"))
print(set("aab") == set("abb"))
print(Counter("aab") == Counter("aba"))
```

In [42]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None
my_guess_line4 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `s1 = "abc"`, `s2 = "bbbca"`: (a) có bao nhiêu cửa sổ độ dài `len(s1)`? (b) liệt kê các cửa sổ; (c) chỉ số bắt đầu đầu tiên của cửa sổ là hoán vị của `s1`.

In [43]:
# Bài 2: điền kết quả tính tay
hand_num_windows = None     # (a) số cửa sổ
hand_windows = None         # (b) list các cửa sổ, theo thứ tự từ trái sang phải
hand_first_index = None     # (c) chỉ số bắt đầu đầu tiên khớp

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 3 (hai mảng đếm, cửa sổ trượt).

In [44]:
def exercise_3_arrays(s1: str, s2: str) -> bool:
    m, n = len(s1), len(s2)
    if m > n:
        return False
    # TODO 1: tạo cnt1 và cnt2, mỗi cái là list 26 số 0
    # TODO 2: đếm s1 vào cnt1 và cửa sổ đầu s2[:m] vào cnt2
    # TODO 3: nếu cnt1 == cnt2 thì trả True
    # TODO 4: với i từ m đến n-1: thêm s2[i], bỏ s2[i-m]; nếu cnt1 == cnt2 thì trả True
    return None


# assert exercise_3_arrays("ab", "eidbaooo") is True
# assert exercise_3_arrays("ab", "eidboaoo") is False
# assert exercise_3_arrays("abc", "ab") is False

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 5, hãy tự viết `exercise_4_two_pointers(s1, s2)` bằng hai con trỏ co giãn với bảng `need`.

In [45]:
def exercise_4_two_pointers(s1: str, s2: str) -> bool:
    # TODO
    pass


# assert exercise_4_two_pointers("ab", "eidbaooo") is True
# assert exercise_4_two_pointers("ab", "eidboaoo") is False
# assert exercise_4_two_pointers("aab", "aaaab") is True

### Bài 5 — Mini challenge: đếm số cửa sổ là hoán vị (khó)

Viết `exercise_5_count_windows(s1, s2)` trả về **có bao nhiêu** vị trí `i` mà `s2[i:i+len(s1)]` là hoán vị của `s1` (các cửa sổ có thể chồng lấn). Nên chạy trong `O(n)`.

```python
exercise_5_count_windows("ab", "abab") -> 3        # ab, ba, ab
exercise_5_count_windows("a", "banana") -> 3
exercise_5_count_windows("abc", "ab") -> 0
```

In [46]:
def exercise_5_count_windows(s1: str, s2: str) -> int:
    # TODO
    pass


# assert exercise_5_count_windows("ab", "abab") == 3
# assert exercise_5_count_windows("a", "banana") == 3
# assert exercise_5_count_windows("abc", "ab") == 0

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `ord` đổi chữ thành mã số; `sorted` xếp chữ; `set` bỏ lặp; `Counter` đếm.
- Mức 2: `ord("d") = 100`, `ord("a") = 97`. `set("aab") = {a, b} = set("abb")`.
- Mức 3: `3`, `True`, `True`, `True`.

**Bài 2**
- Mức 1: số cửa sổ = `n - m + 1`.
- Mức 2: `n = 5`, `m = 3` nên có `3` cửa sổ: `bbb`, `bbc`, `bca`.
- Mức 3: `bca` có đủ `a, b, c` mỗi chữ một lần nên khớp; nó bắt đầu ở chỉ số `2`.

**Bài 3**
- Mức 1: giống mục 7.2.
- Mức 2: `cnt2[ord(s2[i]) - 97] += 1` rồi `cnt2[ord(s2[i - m]) - 97] -= 1`.
- Mức 3: kiểm tra `cnt1 == cnt2` cả **trước** vòng lặp (cửa sổ đầu) và **sau mỗi lần trượt**.

**Bài 4**
- Mức 1: `need` khởi tạo bằng đếm `s1`.
- Mức 2: với mỗi `r`, `need[c] -= 1`, `while need[c] < 0:` trả lại ký tự ở `l` rồi `l += 1`.
- Mức 3: nếu `r - l + 1 == m` thì trả `True`; hết vòng lặp trả `False`.

**Bài 5**
- Mức 1: dùng lại Cách 4 nhưng đếm thay vì trả về sớm.
- Mức 2: đếm một lần cho cửa sổ đầu (nếu `diff == 0`), rồi một lần sau mỗi lần trượt.
- Mức 3: nhớ trả `0` khi `len(s1) > len(s2)`.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [47]:
ans1 = [
    ord("d") - ord("a"),
    sorted("ba") == sorted("ab"),
    set("aab") == set("abb"),
    Counter("aab") == Counter("aba"),
]
print(ans1)
assert ans1 == [3, True, True, True]

[3, True, True, True]


### Đáp án Bài 2

In [48]:
ans2_num = 3
ans2_windows = ["bbb", "bbc", "bca"]
ans2_first = 2
s1, s2 = "abc", "bbbca"
assert len(s2) - len(s1) + 1 == ans2_num
assert [s2[i:i + len(s1)] for i in range(ans2_num)] == ans2_windows
assert first_permutation_index(s1, s2) == ans2_first
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [49]:
def ans3_arrays(s1: str, s2: str) -> bool:
    m, n = len(s1), len(s2)
    if m > n:
        return False
    cnt1 = [0] * 26
    cnt2 = [0] * 26
    for k in range(m):
        cnt1[ord(s1[k]) - 97] += 1
        cnt2[ord(s2[k]) - 97] += 1
    if cnt1 == cnt2:
        return True
    for i in range(m, n):
        cnt2[ord(s2[i]) - 97] += 1
        cnt2[ord(s2[i - m]) - 97] -= 1
        if cnt1 == cnt2:
            return True
    return False

assert ans3_arrays("ab", "eidbaooo") is True
assert ans3_arrays("ab", "eidboaoo") is False
assert ans3_arrays("abc", "ab") is False
rng = random.Random(21)
for _ in range(300):
    a = "".join(rng.choice("abc") for _ in range(rng.randint(1, 5)))
    b = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    assert ans3_arrays(a, b) == oracle(a, b)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [50]:
def ans4_two_pointers(s1: str, s2: str) -> bool:
    m, n = len(s1), len(s2)
    if m > n:
        return False
    need = [0] * 26
    for ch in s1:
        need[ord(ch) - 97] += 1
    l = 0
    for r in range(n):
        c = ord(s2[r]) - 97
        need[c] -= 1
        while need[c] < 0:
            need[ord(s2[l]) - 97] += 1
            l += 1
        if r - l + 1 == m:
            return True
    return False

assert ans4_two_pointers("ab", "eidbaooo") is True
assert ans4_two_pointers("ab", "eidboaoo") is False
assert ans4_two_pointers("aab", "aaaab") is True
rng = random.Random(22)
for _ in range(300):
    a = "".join(rng.choice("abc") for _ in range(rng.randint(1, 5)))
    b = "".join(rng.choice("abc") for _ in range(rng.randint(1, 10)))
    assert ans4_two_pointers(a, b) == oracle(a, b)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

Mỗi lần `diff == 0` (kể cả ở cửa sổ đầu) ta tăng bộ đếm thay vì dừng.

In [51]:
def ans5_count_windows(s1: str, s2: str) -> int:
    m, n = len(s1), len(s2)
    if m > n:
        return 0
    cnt = [0] * 26
    diff = 0

    def change(ch, delta):
        nonlocal diff
        k = ord(ch) - 97
        if cnt[k] == 0:
            diff += 1
        cnt[k] += delta
        if cnt[k] == 0:
            diff -= 1

    for k in range(m):
        change(s1[k], 1)
        change(s2[k], -1)
    total = 1 if diff == 0 else 0
    for i in range(m, n):
        change(s2[i], -1)
        change(s2[i - m], 1)
        if diff == 0:
            total += 1
    return total

assert ans5_count_windows("ab", "abab") == 3
assert ans5_count_windows("a", "banana") == 3
assert ans5_count_windows("abc", "ab") == 0
rng = random.Random(23)
for _ in range(300):
    a = "".join(rng.choice("ab") for _ in range(rng.randint(1, 4)))
    b = "".join(rng.choice("ab") for _ in range(rng.randint(1, 10)))
    m = len(a)
    brute = sum(1 for i in range(len(b) - m + 1) if sorted(b[i:i + m]) == sorted(a))
    assert ans5_count_windows(a, b) == brute
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def checkInclusion(self, s1: str, s2: str) -> bool:
        m, n = len(s1), len(s2)
        if m > n:
            return False
        cnt = [0] * 26                # cnt[c] = đếm trong s1 - đếm trong cửa sổ
        diff = 0                      # số ô cnt khác 0

        def change(ch, delta):
            nonlocal diff
            k = ord(ch) - 97
            if cnt[k] == 0:
                diff += 1             # sắp thành khác 0
            cnt[k] += delta
            if cnt[k] == 0:
                diff -= 1             # vừa về 0

        for k in range(m):            # cửa sổ đầu s2[:m]
            change(s1[k], 1)
            change(s2[k], -1)
        if diff == 0:
            return True
        for i in range(m, n):         # trượt: vào s2[i], ra s2[i-m]
            change(s2[i], -1)
            change(s2[i - m], 1)
            if diff == 0:
                return True
        return False
```

Nhớ ba ý: **hoán vị = cùng bảng đếm**, **cửa sổ độ dài cố định `m`, trượt vào một ra một**, **`diff == 0` nghĩa là khớp**.

### Checklist trước khi submit

- [ ] Đã xử lý `len(s1) > len(s2)` trả `False`.
- [ ] Số cửa sổ là `n - m + 1`; cửa sổ đầu tiên được kiểm tra **trước** vòng trượt.
- [ ] Khi trượt: vào `s2[i]`, ra `s2[i - m]` (không phải `i - m + 1`).
- [ ] Dùng bảng **đếm**, không dùng `set` (mất số lần xuất hiện).
- [ ] Quy tắc cập nhật `diff` nhìn cả trạng thái **trước** (`0` → khác `0`) và **sau** (khác `0` → `0`).
- [ ] Đã thử `("a","a")`, `("ab","a")`, hoán vị ở đầu/cuối `s2`, chữ lặp như `("aab","abbaa")`.
- [ ] Time `O(n + m)`, Space `O(1)` (26 ô).